In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

BIO_LIP_COLUMNS = [
"PDB ID",
"Receptor chain",
"Resolution. '-1.00' stands for lack of resolution information, e.g. for NMR",
"Binding site number code",
"Ligand_ID",
"Ligand_chain",
"Ligand serial number",
"    Binding site residues (with PDB residue numbering)",
"    Binding site residues (with residue re-numbered starting from 1)",
"Catalytic site residues (different sites are separated by ';') (with PDB residue numbering)",
"    Catalytic site residues (different sites are separated by ';') (with residue re-numbered starting from 1)",
"EC number",
"GO terms",
"Binding affinity by manual survey of the original literature. The information in '()' is the PubMed ID",
"Binding affinity provided by the Binding MOAD database. The information in '()' is the ligand information in Binding MOAD",
"Binding affinity provided by the PDBbind-CN database. The information in '()' is the ligand information in PDBbind-CN",
"Binding affinity provided by the BindingDB database",
"UniProt ID",
"PubMed ID",
"Residue sequence number of the ligand (field _atom_site.auth_seq_id in PDBx/mmCIF format)",
"Receptor sequence"]


In [ ]:
df = pd.read_csv('/home/iscb/wolfson/hagairavid/databases/BioLiP_nr.txt', sep="\t", header=None, names=BIO_LIP_COLUMNS)


## Filter relevant columns

In [ ]:
relevant_columns = ["PDB ID", "Receptor chain", "Ligand_ID", "Ligand_chain"]
df = df[relevant_columns].astype('str')
df.shape


## Filter ligands with less than 2 proteins

In [ ]:
# Count occurrences of values in the specified column
value_counts = df["Ligand_ID"].value_counts()

# Get the values that occur more than once
values_to_keep = value_counts[value_counts > 1].index

# Filter the DataFrame to keep rows where the value in the specified column occurs more than once
df = df[df["Ligand_ID"].isin(values_to_keep)]
df = df[df["Ligand_chain"]=='A']
df.shape


## Leave unique protein per ligand

In [ ]:
# filtered_df = df.groupby(["Ligand_ID", "PDB ID"]).apply(lambda x: x.loc[x['Receptor chain'].idxmin()])
filtered_df = df.groupby(["Ligand_ID", "PDB ID"]).apply(lambda x: x.loc[x['Ligand_chain'].idxmin()])
filtered_df.shape


# plot ligand histogram

In [ ]:
column_to_plot = "Ligand_ID"
value_counts = df[column_to_plot].value_counts()

# Plot the histogram
plt.bar(value_counts.index[:100], value_counts.values[:100], color='blue')
plt.xticks(fontsize=6)  # Adjust the font size as per your preference
# Extract the height of the last bar
last_bar_height = value_counts.values[:100][-1]

# Plot a horizontal line parallel to the x-axis at the height of the last bar
plt.axhline(y=last_bar_height, color='red', linestyle='--')
plt.text(-10, last_bar_height, f' y={last_bar_height}', color='red', fontsize=10)


# Rotate x-axis labels for better readability if needed
plt.xticks(rotation=90)

# Add labels and title
plt.xlabel(column_to_plot)
plt.ylabel('Frequency')
plt.title('Histogram of {}'.format(column_to_plot))

# Show plot
plt.show()

In [ ]:
value_counts = df["Ligand_ID"].value_counts()
# Example usage:
pairs_per_ligand = value_counts.apply(lambda x: x*(x-1)//2)
    
# Compute the total number of pairs
total_pairs = pairs_per_ligand.sum()
print("Total number of pairs:", total_pairs)

## Cath relationthip between pairs

In [ ]:
cath_chains = pd.read_csv('/home/iscb/wolfson/hagairavid/databases/cath-domain-list.txt',sep='\s+', names=['pdb_chain', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k'], skiprows=16)
# cath_chains = cath_chains.set_index('pdb_chain')
cath_chains['domain'] = cath_chains['pdb_chain'].str[-2:]
cath_chains['chain'] = cath_chains['pdb_chain'].str[:-2]


In [ ]:

def compare_rows(row1, row2):
    unequal_indices = np.where(row1 != row2)[0]
    if len(unequal_indices) == 0:
        return len(row1)
    else:
        return unequal_indices[0]
    # for i in range(len(row1)):
    #     if row1[i] != row2[i]:
    #         return i
    # return i + 1

In [ ]:
import itertools
import os
import random

matching_columns = []

missing_chains, exist_chains = set(), set()
for id in df["Ligand_ID"].unique()[:500]:
    num_of_pairs = 0
    filtered_rows = df[df['Ligand_ID'] == id]
    # filtered_rows = filtered_rows.groupby(["PDB ID"]).apply(lambda x: x.loc[x['Receptor chain'].idxmin()]) #TODO: check if that correct

    ligand_repo = os.path.join("/home/iscb/wolfson/hagairavid/ligand_alligner/alligned_structures", id)
    os.makedirs(ligand_repo, exist_ok=True)
  
    # Write the filtered rows to a separate file
    num_of_pairs = 1
    unique_pairs = set()
    pairs_file = os.path.join(ligand_repo, f"{id}.txt")
    
    unique_pairs = set()
    with open(pairs_file, "w") as output_file:
        combinations = list(itertools.combinations(filtered_rows.iterrows(), 2))
        random.shuffle(combinations)
        for row1, row2 in combinations:
            row1 = row1[1]
            row2 = row2[1]
            if row1['PDB ID'] != row2['PDB ID'] and (row1[0], row2[0]) not in  unique_pairs and (row2[0], row1[0]) not in  unique_pairs:
                unique_pairs.add((row1[0], row2[0]))
                chain1 = f"{row1['PDB ID']}{row1['Receptor chain']}"
                chain2 = f"{row2['PDB ID']}{row2['Receptor chain']}"
                if chain1 in missing_chains or chain2 in missing_chains:
                    continue
                try:
                    chain1_domains: pd.DataFrame = cath_chains[cath_chains['chain'] == chain1]
                    # row1 = cath_chains[cath_chains['chain'] == idx1].loc[idx1][:8]
                except KeyError:
                    missing_chains.add(chain1)
                    continue
                exist_chains.add(chain1)
                try:    
                    chain2_domains: pd.DataFrame = cath_chains[cath_chains['chain'] == chain2]
                except KeyError:
                    missing_chains.add(chain2)
                    continue
                exist_chains.add(chain2)
                num_of_pairs +=1

                max_similarity = 0
                for _, domain1 in chain1_domains.iterrows():
                    for _, domain2 in chain2_domains.iterrows():
                        max_similarity = max(max_similarity, compare_rows(domain1.values[1:9], domain2.values[1:9]))
                matching_columns.append(max_similarity)
                output_file.write(f"{row1['PDB ID']}:{row1['Receptor chain']} {row2['PDB ID']}:{row2['Receptor chain']} {max_similarity}\n")
                if num_of_pairs > 20:
                    break



In [ ]:
# Plot histogram

print(len(missing_chains))
print(len(exist_chains))
print(len(matching_columns))
plt.hist(matching_columns, bins=range(max(matching_columns) + 2), align='left', edgecolor='black')

plt.xlabel('idx')
plt.ylabel('Frequency')
plt.title('Matching idx between pair of proteins')
plt.xticks(range(max(matching_columns) + 2))
plt.grid(True)
plt.show()